# FlyQuant-1 — Train, export, validate, and publish the model from Google Colab

This notebook uses Colab only as a compute host. FlyQuant's feature engineering, connectome dynamics, training, serialization, and inference remain in the C++20 repository.

The workflow is intentionally reproducible:

1. clone the canonical GitHub repository;
2. build and run the C++ test suite;
3. stage the frozen market/connectome inputs;
4. train the connectome reservoir on TRAIN and evaluate VALIDATION;
5. export `connectome_reservoir_v1.fqmodel`;
6. reload the exact artifact through the production C++ inference path;
7. generate a provenance manifest with hashes and the source commit;
8. push the model + manifest to a dedicated GitHub branch.

**TEST stays locked.** This notebook does not pass `--allow-test`.

## GitHub authentication

In Colab, open **Secrets** (key icon), add a secret named `GITHUB_TOKEN`, and enable notebook access for it. Use a fine-grained token with **Contents: Read and write** permission for the `FruitFly` repository.

The token is read at runtime and is never written to the notebook, git remote, model, or manifest.


In [ ]:
# Notebook parameters
REPO_URL = "https://github.com/RyanAteser/FruitFly.git"
BASE_BRANCH = "main"

MODEL_REL = "models/exports/connectome_reservoir_v1.fqmodel"
MANIFEST_REL = "models/exports/connectome_reservoir_v1.manifest.json"

# Used only for the commit produced by this notebook.
GIT_AUTHOR_NAME = "FlyQuant Colab Export"
GIT_AUTHOR_EMAIL = "flyquant-colab@users.noreply.github.com"


## 1. Clone, build, and test

The export is produced by the same C++ implementation that later loads it. The exact source commit is recorded in the manifest.


In [ ]:
%%bash
set -euo pipefail

cd /content
rm -rf FruitFly

git clone --branch main --single-branch https://github.com/RyanAteser/FruitFly.git
cd FruitFly

cmake -S . -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build -j2
ctest --test-dir build --output-on-failure

mkdir -p input models/exports results
git rev-parse HEAD


## 2. Stage the frozen inputs

Upload these six files into `/content/FruitFly/input/` using the Colab **Files** sidebar:

- `phase1.conf`
- `btc_events.csv`
- `neurons.csv`
- `edges.csv`
- `sensory.csv`
- `outputs.csv`

The next cell checks that every file exists and records its SHA-256. It also creates a runtime copy of the configuration with only `dataset_path` and `results_dir` redirected to the Colab filesystem. Your split timestamps and model hyperparameters are left untouched.


In [ ]:
from pathlib import Path
import hashlib

root = Path("/content/FruitFly")
input_dir = root / "input"

required = [
    "phase1.conf",
    "btc_events.csv",
    "neurons.csv",
    "edges.csv",
    "sensory.csv",
    "outputs.csv",
]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

missing = [name for name in required if not (input_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing input files: " + ", ".join(missing) +
        ". Upload them into /content/FruitFly/input/."
    )

for name in required:
    p = input_dir / name
    print(f"{sha256_file(p)}  {p}")

source_cfg = input_dir / "phase1.conf"
runtime_cfg = input_dir / "phase1.runtime.conf"

lines = source_cfg.read_text().splitlines()
rewritten = []
seen_dataset = False
seen_results = False

for line in lines:
    if line.startswith("dataset_path="):
        rewritten.append("dataset_path=/content/FruitFly/input/btc_events.csv")
        seen_dataset = True
    elif line.startswith("results_dir="):
        rewritten.append("results_dir=/content/FruitFly/results")
        seen_results = True
    else:
        rewritten.append(line)

if not seen_dataset:
    rewritten.append("dataset_path=/content/FruitFly/input/btc_events.csv")
if not seen_results:
    rewritten.append("results_dir=/content/FruitFly/results")

runtime_cfg.write_text("\n".join(rewritten) + "\n")
print(f"\nRuntime config written to {runtime_cfg}")


## 3. Train and export the `.fqmodel`

The authentic connectome remains a fixed recurrent reservoir. Only the constrained UP/DOWN readout is optimized. The artifact is written directly into `models/exports/`.


In [ ]:
%%bash
set -euo pipefail
cd /content/FruitFly

./build/flyquant train-connectome \
  --config input/phase1.runtime.conf \
  --neurons input/neurons.csv \
  --edges input/edges.csv \
  --sensory input/sensory.csv \
  --outputs input/outputs.csv \
  --model-out models/exports/connectome_reservoir_v1.fqmodel \
  --split validation

echo
echo "Exported model:"
ls -lh models/exports/connectome_reservoir_v1.fqmodel
sha256sum models/exports/connectome_reservoir_v1.fqmodel


## 4. Round-trip validation through the production loader

This is not another training pass. It forces the exported artifact back through `predict-connectome`, including the embedded neuron/edge source-hash checks. A mismatch fails the cell.


In [ ]:
%%bash
set -euo pipefail
cd /content/FruitFly

./build/flyquant predict-connectome \
  --config input/phase1.runtime.conf \
  --neurons input/neurons.csv \
  --edges input/edges.csv \
  --model-in models/exports/connectome_reservoir_v1.fqmodel \
  --split validation

echo
echo "Round-trip loader/inference check passed."


## 5. Create a provenance manifest

The manifest binds the export to the exact repository commit, frozen inputs, source config, runtime config, and model hash. It deliberately does **not** include the BTC observations themselves.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess

root = Path("/content/FruitFly")
model = root / MODEL_REL
manifest = root / MANIFEST_REL

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if not model.is_file():
    raise FileNotFoundError(model)

model_bytes = model.stat().st_size

# Keep a safety margin below GitHub's normal 100 MiB per-file hard limit.
if model_bytes >= 95 * 1024 * 1024:
    raise RuntimeError(
        f"{MODEL_REL} is {model_bytes / (1024**2):.1f} MiB. "
        "Do not commit it directly; use Git LFS or a release artifact instead."
    )

source_commit = subprocess.check_output(
    ["git", "-C", str(root), "rev-parse", "HEAD"], text=True
).strip()

tracked_inputs = [
    "phase1.conf",
    "btc_events.csv",
    "neurons.csv",
    "edges.csv",
    "sensory.csv",
    "outputs.csv",
]

payload = {
    "schema": "flyquant-model-export-manifest-v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repository": REPO_URL,
    "base_branch": BASE_BRANCH,
    "source_commit": source_commit,
    "model": {
        "path": MODEL_REL,
        "sha256": sha256_file(model),
        "size_bytes": model_bytes,
        "format": "connectome_reservoir_v1.fqmodel",
    },
    "inputs": {
        name: {
            "sha256": sha256_file(root / "input" / name),
            "size_bytes": (root / "input" / name).stat().st_size,
        }
        for name in tracked_inputs
    },
    "runtime_config": {
        "path": "input/phase1.runtime.conf",
        "sha256": sha256_file(root / "input" / "phase1.runtime.conf"),
    },
    "validation": {
        "round_trip_predict_connectome": "passed",
        "split": "validation",
        "test_used": False,
    },
}

manifest.parent.mkdir(parents=True, exist_ok=True)
manifest.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n")

print(manifest.read_text())


## 6. Publish the export to GitHub

This cell creates a fresh branch named from the model hash, commits only the model + manifest, and pushes that branch to GitHub.

It uses the `GITHUB_TOKEN` Colab secret through a temporary `GIT_ASKPASS` helper. The secret is not added to the remote URL and is deleted from the filesystem immediately after the push.

If a branch for the same model hash already exists, the cell stops instead of overwriting it.


In [ ]:
from google.colab import userdata
from pathlib import Path
import hashlib
import os
import stat
import subprocess

root = Path("/content/FruitFly")
model = root / MODEL_REL
manifest = root / MANIFEST_REL

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise RuntimeError(
        "Colab secret GITHUB_TOKEN is missing. Add it in the Secrets panel "
        "with Contents: Read and write permission for FruitFly."
    )

model_sha = sha256_file(model)
branch = f"model-export/{model_sha[:12]}"

def run(args, *, env=None, capture=False):
    print("+", " ".join(args))
    return subprocess.run(
        args,
        cwd=root,
        env=env,
        check=True,
        text=True,
        capture_output=capture,
    )

# Ensure the working tree is based on the source commit we validated.
run(["git", "checkout", BASE_BRANCH])
run(["git", "status", "--short"])

# Refuse to overwrite an existing local branch.
local = subprocess.run(
    ["git", "show-ref", "--verify", "--quiet", f"refs/heads/{branch}"],
    cwd=root,
)
if local.returncode == 0:
    raise RuntimeError(f"Local branch already exists: {branch}")

run(["git", "checkout", "-b", branch])
run(["git", "config", "user.name", GIT_AUTHOR_NAME])
run(["git", "config", "user.email", GIT_AUTHOR_EMAIL])

run(["git", "add", MODEL_REL, MANIFEST_REL])
run(["git", "commit", "-m", f"Add FlyQuant model export {model_sha[:12]}"])

# Temporary non-interactive credential helper.
askpass = Path("/tmp/flyquant_git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    '  *) echo "" ;;\n'
    'esac\n'
)
askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

try:
    # Check whether that exact export branch is already on the remote.
    probe = subprocess.run(
        ["git", "ls-remote", "--exit-code", "--heads", "origin", branch],
        cwd=root,
        env=env,
        text=True,
        capture_output=True,
    )
    if probe.returncode == 0:
        raise RuntimeError(
            f"Remote branch already exists: {branch}. "
            "The same model hash appears to have been published already."
        )

    run(["git", "push", "-u", "origin", branch], env=env)
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN", None)

repo_slug = REPO_URL.removesuffix(".git").split("github.com/")[-1]
pr_url = f"https://github.com/{repo_slug}/compare/{BASE_BRANCH}...{branch}?expand=1"

print("\nPublished successfully.")
print("Branch:", branch)
print("Model SHA-256:", model_sha)
print("Open a pull request:", pr_url)


## 7. Optional local download

GitHub is now the source of truth for the exported artifact on the model-export branch. If you also want a local copy, the following cell downloads the model and manifest from the Colab runtime.


In [ ]:
from google.colab import files

files.download(f"/content/FruitFly/{MODEL_REL}")
files.download(f"/content/FruitFly/{MANIFEST_REL}")
